# ⚖️ ShieldPay — 02: Preprocessing & Handling Imbalance

Scale features, split data, and apply **SMOTE** to balance the severely skewed class distribution.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from collections import Counter
import warnings, os
warnings.filterwarnings('ignore')
os.makedirs('models', exist_ok=True)

plt.rcParams['figure.facecolor'] = '#0c1118'
plt.rcParams['axes.facecolor']   = '#131b24'
plt.rcParams['axes.edgecolor']   = '#1e2d3d'
plt.rcParams['text.color']       = '#e8edf2'
plt.rcParams['axes.labelcolor']  = '#e8edf2'
plt.rcParams['xtick.color']      = '#6b7a8d'
plt.rcParams['ytick.color']      = '#6b7a8d'
plt.rcParams['grid.color']       = '#1e2d3d'
print('✅ Libraries loaded')

## 1. Load Data

In [ ]:
df = pd.read_csv('creditcard.csv')
print(f'Dataset shape: {df.shape}')
print(f'Class balance: {Counter(df["Class"])}')
X = df.drop('Class', axis=1)
y = df['Class']
print(f'\nFeatures: {X.shape[1]} | Target: Class')

## 2. Scale Amount & Time

`V1–V28` are already PCA-scaled by the dataset provider.  
We only need to scale `Amount` and `Time`.

In [ ]:
scaler = StandardScaler()

# Fit scaler ONLY on training-like data (we'll re-apply after split)
X_scaled = X.copy()
X_scaled[['Time', 'Amount']] = scaler.fit_transform(X[['Time', 'Amount']])

print('Before scaling:')
print(f'  Amount — mean: {X["Amount"].mean():.2f}, std: {X["Amount"].std():.2f}')
print(f'  Time   — mean: {X["Time"].mean():.2f}, std: {X["Time"].std():.2f}')
print()
print('After scaling:')
print(f'  Amount — mean: {X_scaled["Amount"].mean():.4f}, std: {X_scaled["Amount"].std():.4f}')
print(f'  Time   — mean: {X_scaled["Time"].mean():.4f}, std: {X_scaled["Time"].std():.4f}')

## 3. Train / Test Split

Use **stratified** split to preserve the fraud ratio in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # ← keeps fraud % equal in both splits
)

print(f'Train size : {X_train.shape[0]:,} samples')
print(f'Test size  : {X_test.shape[0]:,} samples')
print()
print(f'Train fraud count : {y_train.sum()} ({y_train.mean()*100:.4f}%)')
print(f'Test  fraud count : {y_test.sum()}  ({y_test.mean()*100:.4f}%)')

# Save test set — we NEVER touch this during training
import joblib
joblib.dump((X_test, y_test), 'models/test_set.pkl')
print('\n✅ Test set saved → models/test_set.pkl (do not use for training!)')

## 4. Apply SMOTE

**SMOTE** (Synthetic Minority Over-sampling Technique) generates synthetic fraud samples by interpolating between existing fraud examples.

In [ ]:
print('Before SMOTE:')
print(f'  Legitimate: {Counter(y_train)[0]:,}')
print(f'  Fraud     : {Counter(y_train)[1]:,}')
print()

smote = SMOTE(
    sampling_strategy=0.1,   # fraud will be 10% of majority class (not 50% — avoids over-fitting)
    random_state=42,
    k_neighbors=5
)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print('After SMOTE:')
print(f'  Legitimate: {Counter(y_train_res)[0]:,}')
print(f'  Fraud     : {Counter(y_train_res)[1]:,}')
print(f'  New fraud rate: {y_train_res.mean()*100:.2f}%')

## 5. Compare Sampling Strategies

In [ ]:
strategies = {
    'Original (no resampling)': (X_train, y_train),
}

# SMOTE 10%
sm10 = SMOTE(sampling_strategy=0.1, random_state=42)
X_sm10, y_sm10 = sm10.fit_resample(X_train, y_train)
strategies['SMOTE (10%)'] = (X_sm10, y_sm10)

# SMOTE 50%
sm50 = SMOTE(sampling_strategy=0.5, random_state=42)
X_sm50, y_sm50 = sm50.fit_resample(X_train, y_train)
strategies['SMOTE (50%)'] = (X_sm50, y_sm50)

# Random undersampling
rus = RandomUnderSampler(sampling_strategy=0.1, random_state=42)
X_rus, y_rus = rus.fit_resample(X_train, y_train)
strategies['Undersample (10%)'] = (X_rus, y_rus)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (name, (Xr, yr)) in zip(axes, strategies.items()):
    c = Counter(yr)
    ax.bar(['Legit', 'Fraud'], [c[0], c[1]], color=['#0099ff','#ff4757'], edgecolor='none')
    ax.set_title(name, fontsize=10, fontweight='bold')
    ax.set_ylabel('Count')
    for i, v in enumerate([c[0], c[1]]):
        ax.text(i, v*1.01, f'{v:,}', ha='center', fontsize=9, color='#e8edf2')
plt.suptitle('Sampling Strategy Comparison', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('preprocessing_sampling_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Save Preprocessed Data & Scaler

In [ ]:
import joblib

# Save the scaler — CRITICAL: backend must use this exact scaler
joblib.dump(scaler, 'models/scaler.pkl')
print('✅ Scaler saved → models/scaler.pkl')

# Save resampled training data (SMOTE 10% — best balance of recall vs precision)
joblib.dump((X_train_res, y_train_res), 'models/train_resampled.pkl')
print('✅ Training data saved → models/train_resampled.pkl')

# Save column names for validation in backend
import json
with open('models/feature_columns.json', 'w') as f:
    json.dump(list(X.columns), f)
print('✅ Feature columns saved → models/feature_columns.json')

print()
print('Summary:')
print(f'  Training samples : {len(X_train_res):,}')
print(f'  Test samples     : {len(X_test):,}')
print(f'  Features         : {X_train_res.shape[1]}')
print(f'  Fraud in train   : {y_train_res.sum():,} ({y_train_res.mean()*100:.2f}%)')
print()
print('➡️  Next: 03_Training.ipynb')